In [1]:
# INPUTS:
# -EL_Filename is a string indicating an EyeLink data file from an AX-CPT task in the current path.
#
# OUTPUTS:
# -dfTrials contains information about recording periods (often trials)
# -dfMsg contains information about messages (usually sent from stimulus software)
# -dfFix contains information about fixations
# -dfSacc contains information about saccades
# -dfBlink contains information about blinks
# -dfSamples contains information about individual samples
#
# Created 7/31/18-8/15/18 by DJ.
# Updated 11/12/18 by DJ - switched from "trials" to "recording periods" for experiments with continuous recording
# Adjusted by HH 05/23

In [ ]:
# Import packages
import numpy as np
import pandas as pd
import time
import os
import openpyxl
from shutil import copyfile
import pickle
import cv2

# Set paths
INPUT_DIR = (r"~\Data\Eyetracking_00_Raw_Data")
OUTPUT_DIR = (r"~\Data\Eyetracking_01_Preprocessed_Data")
VIDEO_DIR = (r"~\Lib")

# list = (len(os.listdir(INPUT_DIR)))

# Loop through subdirectories
for file in os.listdir(INPUT_DIR):
    # Check if file ends with .asc
    if file.endswith(".asc"):
        input_file = os.path.join(INPUT_DIR, file)
        
        # Get file prefix from original filename
        EL_FileStart = os.path.splitext(file)[0]
        #folder_name = EL_FileStart[0:3]
        # print(EL_FileStart)
        # print(folder_name)

        DATA_OUTPUT = os.path.join(OUTPUT_DIR, EL_FileStart)
        if not os.path.exists(DATA_OUTPUT):
            os.mkdir(DATA_OUTPUT)

        # Run data processing code on file

        # ===== READ IN FILES ===== #
        # # Navigate to data directory
        # os.chdir(DATA_DIR_INPUT)

        # Read in EyeLink file
        print('Reading in EyeLink file %s...'%input_file)
        t = time.time()
        f = open(input_file,'r')
        fileTxt0 = f.read().splitlines(True) # split into lines
        fileTxt0 = list(filter(None, fileTxt0)) #  remove emptys
        fileTxt0 = np.array(fileTxt0) # concert to np array for simpler indexing
        f.close()
        print('Done! Took %f seconds.'%(time.time()-t))

        # Separate lines into samples and messages
        print('Sorting lines...')
        nLines = len(fileTxt0)
        lineType = np.array(['OTHER']*nLines,dtype='object')
        # Get index of start of recording block
        iStartRec = None
        for i in range(nLines):
            if 'START' in lineType[i]:
                iStartRec = i
                break

        if iStartRec is None:
            iStartRec = 0

        t = time.time()
        for iLine in range(nLines):
            if len(fileTxt0[iLine])<3:
                lineType[iLine] = 'EMPTY'
            elif fileTxt0[iLine].startswith('*') or fileTxt0[iLine].startswith('>>>>>'):
                lineType[iLine] = 'COMMENT'
            elif fileTxt0[iLine].split()[0][0].isdigit() or fileTxt0[iLine].split()[0].startswith('-'):
                lineType[iLine] = 'SAMPLE'
            else:
                lineType[iLine] = fileTxt0[iLine].split()[0]
            if '!CAL' in fileTxt0[iLine]:
                iStartRec = iLine+1
        print('Done! Took %f seconds.'%(time.time()-t))

        # ===== PARSE EYELINK FILE ===== #
        t = time.time()

        # Trials
        print('Parsing recording markers...')
        iNotStart = np.nonzero(lineType!='START')[0]
        dfRecStart = pd.read_csv(input_file,skiprows=iNotStart,header=None,delim_whitespace=True,usecols=[1])
        dfRecStart.columns = ['tStart']
        iNotEnd = np.nonzero(lineType!='END')[0]
        dfRecEnd = pd.read_csv(input_file, skiprows=iNotEnd, header=None, delim_whitespace=True, usecols=[1,5,6])
        dfRecEnd.columns = ['tEnd','xRes','yRes']

        # combine trial info
        dfRec = pd.concat([dfRecStart,dfRecEnd],axis=1)
        nRec = dfRec.shape[0]
        dfTrials = dfRec
        print('%d recording periods found.'%nRec)

        # Import Messages
        print('Parsing stimulus messages...')
        t = time.time()
        iMsg = np.nonzero(lineType=='MSG')[0]
        # set up
        tMsg = []
        txtMsg = []
        t = time.time()
        for i in range(len(iMsg)):
            # separate MSG prefix and timestamp from rest of message
            info = fileTxt0[iMsg[i]].split()
            # extract info
            tMsg.append(int(info[1]))
            txtMsg.append(' '.join(info[2:]))
        # Convert dict to dataframe
        dfMsg = pd.DataFrame({'time':tMsg, 'text':txtMsg})
        print('Done! Took %f seconds.'%(time.time()-t))

        # Import Fixations
        print('Parsing fixations...')
        t = time.time()
        iNotEfix = np.nonzero(lineType!='EFIX')[0]
        dfFix = pd.read_csv(input_file,skiprows=iNotEfix,header=None,delim_whitespace=True,usecols=range(1,8))
        dfFix.columns = ['eye','tStart','tEnd','duration','xAvg','yAvg','pupilAvg']
        # dfFix['t_diff(ms)'] = dfFix['tEnd'] - dfFix['tStart']
        nFix = dfFix.shape[0]
        #print(dfFix.head())  
        print('Done! Took %f seconds.'%(time.time()-t))

        # Saccades
        print('Parsing saccades...')
        t = time.time()
        iNotEsacc = np.nonzero(lineType!='ESACC')[0]
        dfSacc = pd.read_csv(input_file,skiprows=iNotEsacc,header=None,delim_whitespace=True,usecols=range(1,11))
        dfSacc.columns = ['eye','tStart','tEnd','duration','xStart','yStart','xEnd','yEnd','ampDeg','vPeak']
        print('Done! Took %f seconds.'%(time.time()-t))

        # Blinks
        print('Parsing blinks...')
        iNotEblink = np.nonzero(lineType!='EBLINK')[0]
        dfBlink = pd.read_csv(input_file,skiprows=iNotEblink,header=None,delim_whitespace=True,usecols=range(1,5))
        dfBlink.columns = ['eye','tStart','tEnd','duration']
        print('Done! Took %f seconds.'%(time.time()-t))

        # determine sample columns based on eyes recorded in file
        eyesInFile = np.unique(dfFix.eye)
        #print(eyesInFile)
        if len(eyesInFile) > 1:
            print('binocular data detected.')
            cols = ['tSample', 'LX', 'LY', 'LPupil', 'RX', 'RY', 'RPupil']
        else:
            eye = eyesInFile[0]
            print(f'monocular data detected ({eye} eye).')
            cols = ['tSample', f'{eye}X', f'{eye}Y', f'{eye}Pupil']

        # Import samples    
        print('Parsing samples...')
        t = time.time()
        if iStartRec is not None:
            iNotSample = np.nonzero(np.logical_or(lineType!='SAMPLE', np.arange(nLines)<iStartRec))[0]
        else:
            iNotSample = np.nonzero(lineType!='SAMPLE')[0]

        dfSamples = pd.read_csv(input_file,skiprows=iNotSample,header=None,delim_whitespace=True,
                                usecols=range(0,len(cols)))
        dfSamples.columns = cols

        # Convert values to numbers
        for eye in ['L','R']:
            if eye in eyesInFile:
                dfSamples['%cX'%eye] = pd.to_numeric(dfSamples['%cX'%eye],errors='coerce')
                dfSamples['%cY'%eye] = pd.to_numeric(dfSamples['%cY'%eye],errors='coerce')
                dfSamples['%cPupil'%eye] = pd.to_numeric(dfSamples['%cPupil'%eye],errors='coerce')
            else:
                dfSamples['%cX'%eye] = np.nan
                dfSamples['%cY'%eye] = np.nan
                dfSamples['%cPupil'%eye] = np.nan
                
        print('Done! Took %.1f seconds.'%(time.time()-t))


        print('Saving results...')
        t = time.time()
        
        # Make master list of dataframes to write
        allDataFrames = [dfTrials,dfMsg,dfFix,dfSacc,dfBlink] # the dataframes
        allNames = ['Trial','Message','Fixation','Saccade','Blink'] # what they're called

        outFilename = '%s\%s_File.csv'%(DATA_OUTPUT, EL_FileStart)

        # Check if the output file already exists
        if not os.path.exists(outFilename):
            # Output file does not exist, save the dataframes
            for i in range(len(allNames)):
                # Generate the output file name for the specific dataframe
                specificOutFilename = '%s\%s_%s.xlsx'%(DATA_OUTPUT,EL_FileStart, allNames[i])
                outFilename_samples = '%s\%s_Samples.csv'%(DATA_OUTPUT, EL_FileStart)
                
                # Check if the specific output file already exists
                if not os.path.exists(specificOutFilename):
                    # Specific output file does not exist, save the dataframe to Excel
                    print('Saving %s output as %s...' % (allNames[i], specificOutFilename))
                    allDataFrames[i].to_excel(specificOutFilename, float_format='%.1f', index=False, engine='openpyxl')
                else:
                    print('Skipping %s output, file already exists: %s' % (allNames[i], specificOutFilename))

                if not os.path.exists(outFilename_samples):
                    dfSamples.to_csv(outFilename_samples)
                    print('Saving Samples output as %s...'%(outFilename_samples))
                else:
                    print('Skipping output, %s file already exists: ' % (outFilename_samples))
            
        else:
           print('Skipping file %s, files already exist.' % (EL_FileStart)) 




        # Define a dictionary to map movie labels to their corresponding video files
        movie_files = {
            "movie_01": "Charite.mp4", #3724
            "movie_02": "Ziemlich_Beste_Freunde.mp4", #3530
            "movie_03": "High_Seas.mp4", #3807
            "movie_04": "Biohackers.mp4", #3142
            "movie_05": "Downton_Abbey.mp4", #2842
            "movie_06": "New_Amsterdam.mp4" #3453 
        }

        # ===== GET MOVIE TIME STAMPS if this is based on movies ===== #

        if file.endswith('01.asc'):
            #Extract MSG details that pertain to the Movies
            dfcontain_movies = dfMsg[dfMsg['text'].str.contains('movie')]
            dfcontain_movies = dfcontain_movies.astype({"text":str})

            # Adjust columns, clean up
            dfcontain_movies['movie'] = dfcontain_movies['text'].str[-20:]
            dfcontain_movies['val'] = dfcontain_movies['text'].str[:4]
            dfcontain_movies['movie'] = dfcontain_movies['movie'].str.replace('mp4.', '', regex=False)
            dfcontain_movies = dfcontain_movies.drop(['text'], axis=1)

            # ===== CLEAN UP MOVIE TIME STAMPS FILE  ===== #
            # locate the rows in the 'movie' column that contain the desired string, 
            # extract the corresponding time stamp from the 'time' column

            t_start = dfcontain_movies.loc[dfcontain_movies['movie'].str.contains('started'), 'time']
            dft_start = t_start.to_frame()
            dft_start.columns = ['t_start(ms)']                                       # Create new Column Name
            dft_start = dft_start.reset_index(drop=True)                              # Drop Index Row

            t_end = dfcontain_movies.loc[dfcontain_movies['movie'].str.contains('stopped'), 'time']
            dft_end = t_end.to_frame()
            dft_end.columns = ['t_end(ms)']
            dft_end = dft_end.reset_index(drop=True)

            movie_num = dfcontain_movies['movie'].loc[::2]
            movie_num  = movie_num.str[:8]
            dfmovie_num = movie_num.to_frame()
            dfmovie_num.columns = ['movie_num']
            dfmovie_num = dfmovie_num.reset_index(drop=True)
            ORDER = 1  # of 6
            BLOCK = 1  # of 3

            df_block = pd.DataFrame(columns=['Block'])
            df_order = pd.DataFrame(columns=['Order'])
            dft_diff = pd.DataFrame(columns=['t_diff(ms)'])

            for index, row in dfmovie_num.iterrows():
                movie_num = row['movie_num']
                video_file = movie_files[movie_num]
                video_path = os.path.join(VIDEO_DIR, video_file)

                # Open the video file
                cap = cv2.VideoCapture(video_path)
                frame_rate = cap.get(cv2.CAP_PROP_FPS)

                # Calculate the frame timestamps
                total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                length_ms = int((total_frames / frame_rate) * 1000)

                # Release the video capture object
                cap.release()

                # Assign the movie length to the corresponding row in dfhorizontal_stack
                dft_diff.at[index, 't_diff(ms)'] = length_ms
                df_block.at[index, 'Block'] = BLOCK
                df_order.at[index, 'Order'] = ORDER

                # Update BLOCK and ORDER for the next iteration
                ORDER += 1
                if ORDER > 6:
                    ORDER = 1
                    BLOCK += 1
                    if BLOCK > 3:
                        BLOCK = 1


            dfhorizontal_stack = pd.concat([dft_start, dft_end, dfmovie_num, dft_diff, df_block, df_order], ignore_index=False, axis=1)
            dfhorizontal_stack = dfhorizontal_stack.reset_index(drop=True)
            #print(dfhorizontal_stack.head())

            outFilename_movies = '%s\%s_Movie_Timestamps.xlsx'%(DATA_OUTPUT, EL_FileStart)
            dfhorizontal_stack.to_excel(outFilename_movies)

        if file.endswith('02.asc'):
            continue


        else:
            print("Done!")



In [3]:
# #Extract MSG details that pertain to the Movies
# dfcontain_movies = dfMsg[dfMsg['text'].str.contains('movie')]
# dfcontain_movies = dfcontain_movies.astype({"text":str})

# # Adjust columns, clean up
# dfcontain_movies['movie'] = dfcontain_movies['text'].str[-20:]
# dfcontain_movies['val'] = dfcontain_movies['text'].str[:4]
# dfcontain_movies['movie'] = dfcontain_movies['movie'].str.replace('mp4.', '', regex=False)
# dfcontain_movies = dfcontain_movies.drop(['text'], axis=1)

# # ===== CLEAN UP MOVIE TIME STAMPS FILE  ===== #
# # locate the rows in the 'movie' column that contain the desired string, 
# # extract the corresponding time stamp from the 'time' column

# t_start = dfcontain_movies.loc[dfcontain_movies['movie'].str.contains('started'), 'time']
# dft_start = t_start.to_frame()
# dft_start.columns = ['t_start(ms)']                                       # Create new Column Name
# dft_start = dft_start.reset_index(drop=True)                              # Drop Index Row

# t_end = dfcontain_movies.loc[dfcontain_movies['movie'].str.contains('stopped'), 'time']
# dft_end = t_end.to_frame()
# dft_end.columns = ['t_end(ms)']
# dft_end = dft_end.reset_index(drop=True)

# movie_num = dfcontain_movies['movie'].loc[::2]
# movie_num  = movie_num.str[:8]
# dfmovie_num = movie_num.to_frame()
# dfmovie_num.columns = ['movie_num']
# dfmovie_num = dfmovie_num.reset_index(drop=True)

# dft_diff = pd.DataFrame(columns=['t_diff(ms)'], index=range(len(t_start)))
# dft_diff['movie length (ms)'] = dft_end['t_end(ms)'] - dft_start['t_start(ms)']
# # print(dft_diff.head())

# dfhorizontal_stack = pd.concat([dft_start, dft_end, dfmovie_num, dft_diff], ignore_index=False, axis=1)
# dfhorizontal_stack = dfhorizontal_stack.reset_index(drop=True)
# # print(dfhorizontal_stack.head())

# outFilename_movies = '%s\%s_Movie_Timestamps.xlsx'%(DATA_OUTPUT, EL_FileStart)
# dfhorizontal_stack.to_excel(outFilename_movies)